# Notebook for Downloading Events of a Season

### Imports

In [1]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from sportradar_datacore_api.handball import HandballAPI

In [2]:
load_dotenv()  # Load environment variables from .env file if present


True

### Configuration

In [3]:
NAME_COMPETITION = "1. Handball-Bundesliga"

NAME_SEASON = "DAIKIN HBL 2024/25"
YEAR_SEASON = int(NAME_SEASON.split()[-1].split("/")[0])
YEARS_SEASON = NAME_SEASON.split()[-1].replace("/", "-")

PATH_TO_OUTPUT = os.path.join(
    os.getcwd(), "..", "data", "season_24_25"
)

# create path if it does not exist
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

In [4]:
import duckdb
con = duckdb.connect(f'../data/mydb{YEARS_SEASON}.duckdb')

### Initialize API

In [5]:
api = HandballAPI(
    base_url=os.getenv("BASE_URL", ""),
    auth_url=os.getenv("AUTH_URL", ""),
    client_id=os.getenv("CLIENT_ID", ""),
    client_secret=os.getenv("CLIENT_SECRET", ""),
    org_id=os.getenv("CLIENT_ORGANIZATION_ID"),
    scopes=["read:organization"],
    sport="handball",
)

### Get wanted competition ID

In [6]:
id_competition = api.get_competition_id_by_name(NAME_COMPETITION)

# id_competition = int(id_competition)

# Check if the competition was found
if not id_competition:
    raise ValueError(f"Competition '{NAME_COMPETITION}' not found.")
else:
    print(f"→ Competition '{NAME_COMPETITION}' -> {id_competition}")

id_season = api.get_season_id_by_year(
    competition_id=id_competition, season_year=YEAR_SEASON
)
# Check if the season was found
if not id_season:
    raise ValueError(f"Season '{NAME_SEASON}' not found in competition '{NAME_COMPETITION}'.")
else:
    print(f"→ Season '{NAME_SEASON}' -> {id_season}")

→ Competition '1. Handball-Bundesliga' -> 4c445e5c-3956-11ef-9d0e-b74f5c057367
→ Season 'DAIKIN HBL 2024/25' -> cabcf509-4373-11ef-a370-9d3c1e90234a


# Get Teams in the Season

In [7]:
list_entities_season = api.get_teams_by_season_id(season_id=id_season)
# display( pd.json_normalize(list_entities_season[0].to_dict()) )
print(f"→ Number of teams in season '{NAME_SEASON}': {len(list_entities_season)}")

→ Number of teams in season 'DAIKIN HBL 2024/25': 18


### Cols to keep from get_team_by_id

In [8]:
columns_to_keep = [
    "entityId",
    # "organizationId",
    "organization",
    # "entityGroupId",
    # "entityGroup",
    # "internationalReference",
    # "status",
    "nameFullLocal",
    # "additionalNames",
    "nameFullLatin",
    "codeLocal",
    "codeLatin",
    # "address",
    # "social",
    # "contacts",
    # "colors",
    # "historicalNames",
    "externalId",
    # "ageGroup",
    # "gender",
    # "standard",
    # "grade",
    # "representing",
    # "discipline",
    # "updated",
    # "added",
    # "defaultVenueId",
    # "alternateVenueIds",
    # "images"
]

In [9]:
df_teams = pd.DataFrame()

# print len of list_entities_season
print(f"Number of teams in season: {len(list_entities_season)}")

for team in list_entities_season:
    id_team = team.entity_id
    team_details = api.get_team_by_id(entity_id=id_team)
    # print all keys of team_details[0]
    # print("Keys in team_details[0]:")
    # print(json.dumps(list(team_details[0].to_dict().keys()), indent=4, default=str))
    # convert to dataframe
    df_team_details = pd.json_normalize(team_details[0].to_dict())
    # display(entity_details)

    # drop columns that are not in columns list
    df_team_details = df_team_details[[col for col in columns_to_keep if col in df_team_details.columns]]
    # display(df_team_details)
    df_teams = pd.concat([df_teams, df_team_details], ignore_index=True)

    # as json dump
    # print(json.dumps(team_details[0].to_dict(), indent=4, default=str))

Number of teams in season: 18


In [10]:
# Drop tables if exist
con.execute("DROP TABLE IF EXISTS teams")
# Create duckdb table
con.execute("""
CREATE TABLE IF NOT EXISTS teams AS SELECT * FROM df_teams
""")

In [11]:
# plot table
con.execute("SELECT * FROM teams").df()

,entityId,nameFullLocal,nameFullLatin,codeLocal,codeLatin,externalId
0,febb3114-3952-11ef-b6f7-af5c55c3771d,TVB Stuttgart,TVB Stuttgart,TVB,TVB,17
1,fe911367-3952-11ef-9131-af5c55c3771d,VfL Gummersbach,VfL Gummersbach,GUM,GUM,7
2,fe8d1885-3952-11ef-9130-af5c55c3771d,HC Erlangen,HC Erlangen,HCE,HCE,6
3,fe73c376-3952-11ef-8a18-af5c55c3771d,HSG Wetzlar,HSG Wetzlar,WET,WET,1
4,ffe84692-3952-11ef-954e-af5c55c3771d,1. VfL Potsdam,1. VfL Potsdam,POT,POT,93
5,fe848316-3952-11ef-8185-af5c55c3771d,SC Magdeburg,SC Magdeburg,SCM,SCM,4
6,fe80598a-3952-11ef-914c-af5c55c3771d,Rhein-Neckar Löwen,Rhein-Neckar Löwen,RNL,RNL,3
7,fe88f93b-3952-11ef-aa5d-af5c55c3771d,SG Flensburg-Handewitt,SG Flensburg-Handewitt,SGF,SGF,5
8,fe7bdd16-3952-11ef-b585-af5c55c3771d,Füchse Berlin,Füchse Berlin,BER,BER,2
9,fef51771-3952-11ef-97b4-af5c55c3771d,ThSV Eisenach,ThSV Eisenach,EIS,EIS,32


In [12]:
columns_to_keep_fixtures = [
    "fixtureId",
    # "organizationId",
    # "organization",
    "seasonId",
    # "season",
    # "practiceDrillType",
    # "internationalReference",
    # "status",
    "fixtureNumber",
    "nameLocal",
    "nameLatin",
    "startTimeLocal",
    "startTimeUTC",
    # "startTimeActualUTC",
    # "endTimeActualUTC",
    # "timesUnconfirmed",
    # "locked",
    # "placingIfWon",
    # "placingIfLost",
    # "attendance",
    # "sellout",
    # "duration",
    # "durationFull",
    # "ticketURL",
    # "stageCode",
    # "stage",
    # "seriesCode",
    # "poolCode",
    # "roundCode",
    # "round",
    "roundNumber",
    # "liveDataAvailable",
    # "liveVideoAvailable",
    # "fixtureType",
    # "maximumPeriodTypeUsed",
    # "competitorType",
    "competitors",
    # "venueId",
    # "venue",
    "externalId",
    # "profileId",
    # "includeInStandings",
    # "updated",
    # "added",
    # "seriesFixtureNumber",
    # "discipline",
    # "broadcasts"
]

columns_to_keep_competitors = [
    "entityId",
    # "conferenceId",
    # "divisionId",
    # "includeInConferenceStatistics",
    "isHome",
    # "includeInRepresentation",
    "draw",
    # "resultStatus",
    "resultPlace",
    # "resultSecondaryScorePlace",
    # "startingNumber",
    "score",
    # "secondaryScore",
    # "shootOutAttempts",
    # "rosterStatus",
    # "isNeutralVenue",
    # "uniformId",
    # "externalId",
]

In [22]:
import pandas as pd
from collections import defaultdict

def _standings_table(stats: dict) -> pd.DataFrame:
    """Build a standings table from the current stats dict."""
    if not stats:
        return pd.DataFrame(columns=["team","Pts","GD","GF","GA","W","D","L","P"])
    rows = []
    for team, s in stats.items():
        rows.append({
            "team": team,
            "Pts": s["Pts"],
            "GD": s["GF"] - s["GA"],
            "GF": s["GF"],
            "GA": s["GA"],
            "W":  s["W"],
            "D":  s["D"],
            "L":  s["L"],
            "P":  s["P"],
        })
    st = pd.DataFrame(rows)
    # Rank: Points desc, GD desc, GF desc, then team name asc as deterministic tie-breaker
    st = st.sort_values(["Pts","GD","GF","team"], ascending=[False, False, False, True], kind="mergesort")
    st["rank"] = range(1, len(st) + 1)
    return st

def _ensure_team(stats: dict, team: str):
    if team not in stats:
        stats[team] = defaultdict(int)  # keys: P,W,D,L,GF,GA,Pts

def add_standing_columns(df_fixture: pd.DataFrame, *, points_win: int = 2, points_draw: int = 1) -> pd.DataFrame:
    """
    Adds standing_before/after columns for both home and away teams by iterating fixtures chronologically.
    Expects columns: name_team_home, name_team_away, score_home, score_away, startTimeUTC (or startTimeLocal).
    """
    df = df_fixture.copy()

    # Choose a time column; parse and sort
    time_col = "startTimeUTC" if "startTimeUTC" in df.columns else "startTimeLocal"
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.sort_values([time_col, "fixtureNumber"]).reset_index(drop=True)

    # Prepare output columns
    for col in [
        "standing_before_game_home","standing_before_game_away",
        "standing_after_game_home","standing_after_game_away"
    ]:
        if col not in df.columns:
            df[col] = pd.NA

    # Running stats
    stats = {}

    # Iterate fixtures
    for idx, row in df.iterrows():
        home = row["name_team_home"]
        away = row["name_team_away"]
        sh = pd.to_numeric(row["score_home"], errors="coerce")
        sa = pd.to_numeric(row["score_away"], errors="coerce")

        # Skip fixtures with missing teams/scores
        if pd.isna(sh) or pd.isna(sa) or not isinstance(home, str) or not isinstance(away, str):
            continue

        # Ensure both teams exist in the running table
        _ensure_team(stats, home)
        _ensure_team(stats, away)

        # --- standings BEFORE this game ---
        st_before = _standings_table(stats).set_index("team")["rank"]
        df.at[idx, "standing_before_game_home"] = int(st_before.get(home, len(st_before) + 1))
        df.at[idx, "standing_before_game_away"] = int(st_before.get(away, len(st_before) + 1))

        # --- update running stats with this game's result ---
        # matches played & goals
        stats[home]["P"] += 1
        stats[away]["P"] += 1
        stats[home]["GF"] += int(sh)
        stats[home]["GA"] += int(sa)
        stats[away]["GF"] += int(sa)
        stats[away]["GA"] += int(sh)

        if sh > sa:
            stats[home]["W"] += 1
            stats[away]["L"] += 1
            stats[home]["Pts"] += points_win
        elif sh < sa:
            stats[away]["W"] += 1
            stats[home]["L"] += 1
            stats[away]["Pts"] += points_win
        else:
            stats[home]["D"] += 1
            stats[away]["D"] += 1
            stats[home]["Pts"] += points_draw
            stats[away]["Pts"] += points_draw

        # --- standings AFTER this game ---
        st_after = _standings_table(stats).set_index("team")["rank"]
        df.at[idx, "standing_after_game_home"] = int(st_after[home])
        df.at[idx, "standing_after_game_away"] = int(st_after[away])

    # Restore original ordering if you like:
    df = df.sort_values("fixtureNumber").reset_index(drop=True)
    return df


### Get the fixtures (matches) of a season and insert to duckdb

In [ ]:
list_fixtures = api.get_list_matches_by_season_id(season_id=id_season)

print(f"Found {len(list_fixtures)} fixtures.")

# drop table fixtures if exists
con.execute("DROP TABLE IF EXISTS fixtures")

for match in list_fixtures:
    df_fixture = pd.json_normalize(match.to_dict())
    # drop columns that are not in columns list
    df_fixture = df_fixture[
        [col for col in columns_to_keep_fixtures if col in df_fixture.columns]
    ]

    competitors_expanded = pd.json_normalize(
        df_fixture["competitors"].explode().to_list()
    )
    # drop columns that are not in columns list
    competitors_expanded = competitors_expanded[
        [col for col in columns_to_keep_competitors if col in competitors_expanded.columns]
    ]    
    # insert names to competitors_expanded
    competitors_expanded = competitors_expanded.merge(
        df_teams[["entityId", "nameFullLocal"]],
        left_on="entityId",
        right_on="entityId",
        how="left",
    )
    # display(competitors_expanded)
    # convert competitors_expanded to json and add to df_fixture
    df_fixture = df_fixture.drop(columns=["competitors"])
    df_fixture = df_fixture.assign(competitors= [competitors_expanded.to_dict(orient="records")])
    # insert entityId_home and entityId_away to df_fixture
    df_fixture = df_fixture.assign(
        entityId_home=competitors_expanded[competitors_expanded["isHome"] == True]["entityId"].values[0],
        entityId_away=competitors_expanded[competitors_expanded["isHome"] == False]["entityId"].values[0],
    )
    # insert name_team_home and name_team_away to df_fixture
    df_fixture = df_fixture.assign(
        name_team_home=competitors_expanded[competitors_expanded["isHome"] == True]["nameFullLocal"].values[0],
        name_team_away=competitors_expanded[competitors_expanded["isHome"] == False]["nameFullLocal"].values[0],
    )
    # insert score_home and score_away to df_fixture
    df_fixture = df_fixture.assign(
        score_home=competitors_expanded[competitors_expanded["isHome"] == True]["score"].values[0],
        score_away=competitors_expanded[competitors_expanded["isHome"] == False]["score"].values[0],
    )
    # insert resultPlace_home and resultPlace_away to df_fixture
    df_fixture = df_fixture.assign(
        resultPlace_home=competitors_expanded[competitors_expanded["isHome"] == True]["resultPlace"].values[0],
        resultPlace_away=competitors_expanded[competitors_expanded["isHome"] == False]["resultPlace"].values[0],
    )

    # drop col competitors
    df_fixture = df_fixture.drop(columns=["competitors"])

    # calc total_goals_home and total_goals_away
    df_fixture = df_fixture.assign(
        total_goals_home=pd.to_numeric(df_fixture["score_home"], errors="coerce"),
        total_goals_away=pd.to_numeric(df_fixture["score_away"], errors="coerce"),
    )

    goals_home = df_fixture.groupby("name_team_home")["total_goals_home"].sum()
    goals_away = df_fixture.groupby("name_team_away")["total_goals_away"].sum()

    # Combine home and away goals for each team
    df_goals = pd.DataFrame({
        "team": pd.concat([goals_home, goals_away]).index.unique(),
        "goals_home": goals_home,
        "goals_away": goals_away
    }).fillna(0)

    df_goals["goals_total"] = df_goals["goals_home"] + df_goals["goals_away"]

    display(df_goals)



    # display(df_fixture)
    # break
    # append to duckdb table
    con.execute(
        """
    CREATE TABLE IF NOT EXISTS fixtures AS SELECT * FROM df_fixture
    """
    )
    con.execute(
        """
    INSERT INTO fixtures SELECT * FROM df_fixture
    """
    )



# plot duplicate fixtureid's
display(
    con.execute(
        """
SELECT fixtureId, COUNT(*) as count FROM fixtures
GROUP BY fixtureId
HAVING count > 1
"""
    ).df()
)

# drop duplicate fixtureid's keeping first
con.execute(
    """
DELETE FROM fixtures
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM fixtures
    GROUP BY fixtureId
)
"""
)
# plot table
display(con.execute("SELECT * FROM fixtures").df())

# describe table fixtures
display(con.execute("DESCRIBE fixtures").df())

Found 306 fixtures.


,fixtureId,count
0,00c08679-4374-11ef-80bd-73cf0bc66b45,2


,fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,competitors,...,name_team_home,name_team_away,score_home,score_away,resultPlace_home,resultPlace_away,standing_before_game_home,standing_before_game_away,standing_after_game_home,standing_after_game_away
0,00c08679-4374-11ef-80bd-73cf0bc66b45,cabcf509-4373-11ef-a370-9d3c1e90234a,61,TSV Hannover-Burgdorf vs. SG Flensburg-Handewitt,<NA>,2024-10-20T15:00:00,2024-10-20 13:00:00,7,57979,[{'entityId': 'fe88f93b-3952-11ef-aa5d-af5c55c...,...,TSV Hannover-Burgdorf,SG Flensburg-Handewitt,31,30,1,2,2,1,1,2
1,00febf5a-4374-11ef-9a3c-a3f6150225cd,cabcf509-4373-11ef-a370-9d3c1e90234a,62,SC Magdeburg vs. SC DHfK Leipzig,<NA>,2024-10-20T16:00:00,2024-10-20 14:00:00,7,57980,[{'entityId': 'fe848316-3952-11ef-8185-af5c55c...,...,SC Magdeburg,SC DHfK Leipzig,35,29,1,2,2,1,1,2
2,0182fb4e-4374-11ef-a879-691708cc0833,cabcf509-4373-11ef-a370-9d3c1e90234a,63,TBV Lemgo Lippe vs. TVB Stuttgart,<NA>,2024-10-20T16:30:00,2024-10-20 14:30:00,7,57981,[{'entityId': 'fe99a935-3952-11ef-9dd4-af5c55c...,...,TBV Lemgo Lippe,TVB Stuttgart,28,24,1,2,1,2,1,2
3,037982a4-4374-11ef-982a-1f74ee999933,cabcf509-4373-11ef-a370-9d3c1e90234a,64,SC DHfK Leipzig vs. MT Melsungen,<NA>,2024-10-24T19:00:00,2024-10-24 17:00:00,8,57982,[{'entityId': 'feace61d-3952-11ef-ae23-af5c55c...,...,SC DHfK Leipzig,MT Melsungen,27,28,2,1,2,1,2,1
4,03a7d8f5-4374-11ef-abc8-75b52ec025f2,cabcf509-4373-11ef-a370-9d3c1e90234a,65,Handball Sport Verein Hamburg vs. TSV Hannover...,<NA>,2024-10-24T19:00:00,2024-10-24 17:00:00,8,57983,[{'entityId': '0045fbc4-3953-11ef-a217-af5c55c...,...,Handball Sport Verein Hamburg,TSV Hannover-Burgdorf,32,32,1,1,1,2,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,fd35640d-4373-11ef-96cf-3d96bea3d744,cabcf509-4373-11ef-a370-9d3c1e90234a,56,Rhein-Neckar Löwen vs. HC Erlangen,<NA>,2024-10-17T19:00:00,2024-10-17 17:00:00,7,57974,[{'entityId': 'fe80598a-3952-11ef-914c-af5c55c...,...,Rhein-Neckar Löwen,HC Erlangen,38,33,1,2,2,1,1,2
302,fd5a003d-4373-11ef-9920-89956fae12b0,cabcf509-4373-11ef-a370-9d3c1e90234a,57,VfL Gummersbach vs. ThSV Eisenach,<NA>,2024-10-18T19:00:00,2024-10-18 17:00:00,7,57975,[{'entityId': 'fe911367-3952-11ef-9131-af5c55c...,...,VfL Gummersbach,ThSV Eisenach,34,32,1,2,2,1,1,2
303,fef1685c-4373-11ef-94c0-099f83d5fd51,cabcf509-4373-11ef-a370-9d3c1e90234a,59,MT Melsungen vs. Füchse Berlin,<NA>,2024-10-19T19:00:00,2024-10-19 17:00:00,7,57977,[{'entityId': 'fe7bdd16-3952-11ef-b585-af5c55c...,...,MT Melsungen,Füchse Berlin,33,31,1,2,2,1,1,2
304,ff1398a1-4373-11ef-92f0-fdc5f2a62d4e,cabcf509-4373-11ef-a370-9d3c1e90234a,58,FRISCH AUF! Göppingen vs. SG BBM Bietigheim,<NA>,2024-10-18T20:00:00,2024-10-18 18:00:00,7,57976,[{'entityId': 'fea4237d-3952-11ef-9fcd-af5c55c...,...,FRISCH AUF! Göppingen,SG BBM Bietigheim,30,25,1,2,1,2,1,2


,column_name,column_type,null,key,default,extra
0,fixtureId,VARCHAR,YES,None,None,None
1,seasonId,VARCHAR,YES,None,None,None
2,fixtureNumber,BIGINT,YES,None,None,None
3,nameLocal,VARCHAR,YES,None,None,None
4,nameLatin,INTEGER,YES,None,None,None
5,startTimeLocal,VARCHAR,YES,None,None,None
6,startTimeUTC,TIMESTAMP_NS,YES,None,None,None
7,roundNumber,VARCHAR,YES,None,None,None
8,externalId,VARCHAR,YES,None,None,None
9,competitors,"STRUCT(entityId VARCHAR, isHome BOOLEAN, draw ...",YES,None,None,None


### Get the details of one fixture

In [14]:
# list_fixture_details = api.get_fixture_by_id(fixture_id=id_fixture)
# fixture_details = list_fixture_details[0]
# display( pd.json_normalize(fixture_details.to_dict()))
# print(json.dumps(fixture_details.to_dict(), indent=4, default=str))

In [15]:
list_team_ids = [comp.entity_id for comp in fixture_details.competitors]

for tid in list_team_ids:
    team = api.get_team_by_id(tid)
    if not team:
        continue
    team = team[0]
    display(pd.DataFrame([team.to_dict()]))

    


NameError: name 'fixture_details' is not defined

### Get fixture

In [ ]:

# con.execute("CREATE TABLE IF NOT EXISTS events;")

In [ ]:
events = api.get_fixture_events_by_id(
    list_fixtures[0].fixture_id, setup_only=True, with_scores=True
)
print(f"  Events count: {len(events)}")


for event in events:
    if "data" in event and event["data"]:
        # Merge the data dict into the event dict
        event.update(event["data"])
        del event["data"]  # Remove the original data field
        pass
    if "options" in event and event["options"]:
        # Merge the options dict into the event dict
        event.update(event["options"])
        del event["options"]  # Remove the original options field


cols = [
    # "clientId",
    # "clientType",
    "fixtureId",
    # "organizationId",
    # "received",
    # "sport",
    # "topic",
    # "type",
    "class",
    "eventId",
    "eventTime",
    "eventType",
    "subType",
    # "timestamp",
    "attendance",
    # "numberOfPeriods",
    # "periodLength",
    "entityId",
    "personId",
    # "status",
    # "active",
    "bib",
    # "captain",
    "name",
    "position",
    # "starter",
    # "number",
    "scores",
    "periodId",
    # "sequence",
    "playId",
    "clock",
    "success",
    "x",
    "y",
    "attackType",
    "goalKeeperId",
    "location",
    # "options",
    "failureReason",
    # "flagged",
    # "value",
    "emptyNet",
]


df = pd.DataFrame(events, columns=cols)
display(df)
print(f"Columns: {df.columns.tolist()}")

In [ ]:
list_player_ids = df['personId'].dropna().unique().tolist()
list_team_ids = df['entityId'].dropna().unique().tolist()

for tid in list_team_ids:
    team = api.get_team_by_id(tid)
    if not team:
        continue
    team = team[0]
    display(pd.DataFrame(team.to_dict()))

for pid in list_player_ids:
    player = api.get_player_by_id(pid)
    if not player:
        continue
    player = player[0]
    display(player)
    


### Save to CSV

In [ ]:
api.save_events_to_csv(
    events,
    file_path=f"fixture_{list_fixtures[0].fixture_id}_events.csv",
)